# Twist 5: pedestrian speed and clearance time (demo notebook)

**Input:** one video of the crossing.
**Output:** the same video with a box on every person, their **ID** and their **speed in m/s**, plus a table of speeds and the **clearance time**.

This one notebook replaces the separate Lesson 4 and Lesson 5 labs for our twist. It keeps what T5 needs from each and leaves out the rest.

| Taken from | What is used here |
|---|---|
| **Lesson 4** (deployment) | Load the model once. Refuse bad input with a clear message. One function that does the whole job. A Gradio page with a public link. A table of error tests. |
| **Lesson 5, Lab B** (tracking) | ByteTrack to give every person an ID. The feet point. Crossing a line. The tracker settings sweep. The path picture for checking tracks. |
| **Lesson 5, marking regions** | Reading pixel positions off a grid. Two marks written as two points each. The check that the camera did not move. The cross-product test for which side of a mark a point is on. |

**Left out on purpose:** everything for T4 (waiting area, dwell time), the cold-start timing and cloud-or-edge discussion from Lesson 4, and the drawn sample clip and its truth scoring.

## How the speed is calculated

1. The detector finds each person in each frame (box + direction class).
2. ByteTrack links the boxes over time, so each person keeps one ID.
3. For each person we find the exact moment their **feet** cross **mark A** and **mark B**.
4. **Speed = D / (time at B − time at A)**, where **D** is the distance between the two marks, measured with a tape.
5. **Clearance time = W / slowest walking speed**, where **W** is the walkway length (kerb to kerb). We use the slowest walker, not the average.

No pixel-to-metre conversion is needed, because D is measured in metres on the ground.

## The order of work

1. Setup
2. Look at your clip, check the camera did not move, and mark A and B by clicking
3. Settings (the only cell you edit)
4. Load the model once
5. Check the input before using it
6. Geometry: when did the feet cross a mark
7. Tracking
8. Speed and clearance time
9. Draw the output video
10. The whole service in one function
11. Run it on your clip
12. Check the tracks by looking at them
13. A web page with a public link
14. Error tests
15. Tune the tracker (optional)
16. Before leaving

---

## 1. Setup

**What this cell does:** installs the libraries when running on Colab. Ultralytics is pinned to the same version as Lessons 3 to 5, because the weights and the library must agree. `lap` is what ByteTrack uses to match boxes. `gradio` builds the web page in Section 13. Outside Colab nothing is installed, and the cell only reports where it is running.

In [ ]:
ULTRALYTICS_VERSION = "8.4.120"

try:
    import google.colab  # noqa: F401
    RUNNING_ON_COLAB = True
except ImportError:
    RUNNING_ON_COLAB = False

if RUNNING_ON_COLAB:
    import subprocess
    subprocess.run(
        ["pip", "install", "-q", "ultralytics==" + ULTRALYTICS_VERSION, "lap", "gradio"],
        check=True,
    )

print("Running on Colab:", RUNNING_ON_COLAB)

**What this cell does:** imports everything the notebook uses, mounts Google Drive on Colab, and works out the file locations:

- `WEIGHTS`: the Lesson 3 model, `artifacts/L3/best.pt`
- `VIDEO_IN`: your clip, `data/clips/walkway.mp4`
- `WORK_DIR`: where this notebook saves its results, `artifacts/T5/`
- `VIDEO_OUT`: the finished demo video

In [ ]:
import csv
import json
import os
import shutil
import subprocess
import time
from collections import Counter

import cv2
import numpy as np
import yaml
from PIL import Image, ImageDraw
from ultralytics import YOLO
from ultralytics.utils import ROOT as ULTRALYTICS_ROOT

if RUNNING_ON_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_ROOT = "/content/drive/MyDrive/C389"
else:
    PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))

WEIGHTS = os.path.join(PROJECT_ROOT, "artifacts", "L3", "best.pt")
VIDEO_IN = os.path.join(PROJECT_ROOT, "data", "clips", "walkway.mp4")
WORK_DIR = os.path.join(PROJECT_ROOT, "artifacts", "T5")
VIDEO_OUT = os.path.join(WORK_DIR, "t5_demo_output.mp4")
os.makedirs(WORK_DIR, exist_ok=True)

print("weights  :", WEIGHTS)
print("video in :", VIDEO_IN)
print("work dir :", WORK_DIR)

---

## 2. Look at your clip, check the camera, and mark A and B

A mark has to be written down in **pixels**, counted from the top left corner of the frame: `x` across, `y` down.

**What this cell does:**

- `video_info` reads the frame rate, number of frames and size of a video.
- `read_frame` pulls one frame out of a video.
- `with_grid` draws a labelled grid (every 50 pixels) on a picture, so you can read positions by eye.
- It then opens your clip, prints its details, saves the first frame as `first_frame.png`, and shows the first frame with the grid.

The grid is the fallback when clicking is not available (it works on Colab only). Write down the pixel positions of **two points on mark A** and **two on mark B**: the two ends of each line you chalked across the crossing. For a more exact reading, open `first_frame.png` in Paint or Preview, which shows the pixel under the pointer.

In [ ]:
def video_info(path):
    """Frame rate, frame count and size of a video, or None when OpenCV cannot open it."""
    video = cv2.VideoCapture(path)
    if not video.isOpened():
        video.release()
        return None
    info = {
        "fps": video.get(cv2.CAP_PROP_FPS),
        "frames": int(video.get(cv2.CAP_PROP_FRAME_COUNT)),
        "width": int(video.get(cv2.CAP_PROP_FRAME_WIDTH)),
        "height": int(video.get(cv2.CAP_PROP_FRAME_HEIGHT)),
    }
    video.release()
    return info


def read_frame(path, index):
    """One frame of a video, as OpenCV returns it: a BGR array."""
    video = cv2.VideoCapture(path)
    video.set(cv2.CAP_PROP_POS_FRAMES, index)
    ok, frame = video.read()
    video.release()
    if not ok:
        raise ValueError("Could not read frame %d of %s" % (index, path))
    return frame


def as_picture(frame):
    """An OpenCV frame as a PIL image, which is what a notebook displays."""
    return Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))


def with_grid(picture, step=50):
    """The picture with a labelled pixel grid drawn over it."""
    canvas = picture.copy()
    draw = ImageDraw.Draw(canvas)
    for x in range(0, canvas.width, step):
        draw.line([(x, 0), (x, canvas.height)], fill=(0, 150, 220), width=1)
        if x % (2 * step) == 0:
            draw.text((x + 2, 2), str(x), fill=(0, 90, 160))
    for y in range(0, canvas.height, step):
        draw.line([(0, y), (canvas.width, y)], fill=(0, 150, 220), width=1)
        if y % (2 * step) == 0:
            draw.text((2, y + 2), str(y), fill=(0, 90, 160))
    return canvas


CLIP = video_info(VIDEO_IN) if os.path.exists(VIDEO_IN) else None
if CLIP is None:
    raise FileNotFoundError(
        "No readable video at %s.\n"
        "Upload your clip to Drive as data/clips/walkway.mp4, or change VIDEO_IN above."
        % VIDEO_IN)

print("frames : %d at %.2f per second, %.1f seconds"
      % (CLIP["frames"], CLIP["fps"], CLIP["frames"] / CLIP["fps"]))
print("size   : %d x %d pixels" % (CLIP["width"], CLIP["height"]))

FIRST_FRAME = as_picture(read_frame(VIDEO_IN, 0))
FIRST_FRAME.save(os.path.join(WORK_DIR, "first_frame.png"))
with_grid(FIRST_FRAME)

**Check the camera did not move.** Marks are read off the **first** frame, and they only mean a place on the ground while the camera stays exactly where it was. A phone held in the hand drifts, and a drift of a few pixels is enough to move a mark off its line.

**What this cell does:** compares the **first and last frames** pixel by pixel in grey. People walking through change only a small part of the picture, so the **median** difference stays near zero while the camera is still. A camera that moved shifts everything, and the median jumps. It prints the median and shows a picture of where the two frames differ: people only (fine) or edges everywhere (camera moved).

As a guide, a median of a few units is compression noise or a change in light. Above `MOVED_ABOVE` (8) means look at the picture before trusting any speed. If it is high, film again with the phone on a tripod, a clamp or a wall. A high value cannot be repaired afterwards.

In [ ]:
MOVED_ABOVE = 8


def frame_difference(first, second):
    """The median difference of two frames in grey, and a picture of where they differ."""
    grey_first = cv2.cvtColor(first, cv2.COLOR_BGR2GRAY)
    grey_second = cv2.cvtColor(second, cv2.COLOR_BGR2GRAY)
    difference = cv2.absdiff(grey_first, grey_second)
    return float(np.median(difference)), Image.fromarray(255 - difference)


median, difference_picture = frame_difference(read_frame(VIDEO_IN, 0),
                                              read_frame(VIDEO_IN, CLIP["frames"] - 1))
print("median difference, first frame against last : %.1f" % median)
if median > MOVED_ABOVE:
    print("That is high. Look at the picture: edges everywhere mean the camera moved,")
    print("and every mark read off the first frame is in the wrong place by the end.")
else:
    print("Low. The camera stayed where it was, as far as this check can tell.")
difference_picture

### Mark A and mark B by clicking (easier and more accurate than reading the grid)

Each mark is a line on the ground **across** the walking direction: a stripe edge, a chalk line or a piece of tape. You tell the notebook where it is by clicking two points on it.

**What this cell does (Colab only):** shows the first frame of your clip as a clickable picture. You click **four points**:

1. **Mark A, point 1:** on the line, at the end **nearest the camera**.
2. **Mark A, point 2:** on the same line, at the end **farthest from the camera**.
3. **Mark B, point 1** and **point 2:** the same, for the second line.

How to click well:

- Click **right on the line**. Zoom the browser (Ctrl and +) if you need to. The cross-hair and the `x, y` readout show exactly where you are.
- Put the two points of a mark **as far apart as you can**. A line drawn through two close points can point slightly the wrong way, and the error grows towards the edges of the picture.
- Look at the **long red line (A) and blue line (B)**. They are drawn right across the picture, and this is the line the notebook uses to decide when someone crossed. It must follow the real mark along its whole length, not only between your two points.
- Use **Undo last point** or **Start again** if a click is off.

Your points are remembered by the notebook. Run the settings cell next and it picks them up. **Tip for the next clip:** lay bright tape or chalk lines across the ground before filming, so the marks are easy to see and click.

In [ ]:
import base64
import io
import uuid

CLICKED_POINTS = []

CLICK_PAGE = """
<div style="font-family:sans-serif;max-width:__SHOW__px">
  <div id="__UID__m" style="font-size:15px;font-weight:bold;margin:6px 0"></div>
  <canvas id="__UID__c" width="__W__" height="__H__"
          style="width:100%;height:auto;cursor:crosshair;border:1px solid #888"></canvas>
  <div style="margin-top:6px">
    <button id="__UID__u">Undo last point</button>
    <button id="__UID__r">Start again</button>
    <span id="__UID__p" style="margin-left:12px;color:#555"></span>
  </div>
</div>
<script>
(function () {
  var uid = "__UID__", W = __W__, H = __H__;
  var canvas = document.getElementById(uid + "c"), ctx = canvas.getContext("2d");
  var msg = document.getElementById(uid + "m"), pos = document.getElementById(uid + "p");
  var img = new Image(), pts = [], mouse = null;
  var COLOURS = ["#d62828", "#d62828", "#1d4ed8", "#1d4ed8"];
  var PROMPTS = [
    "Mark A, point 1: click on the line, at the end NEAREST the camera",
    "Mark A, point 2: click on the same line, at the end FARTHEST from the camera",
    "Mark B, point 1: click on the line, at the end NEAREST the camera",
    "Mark B, point 2: click on the same line, at the end FARTHEST from the camera",
    "Done. Check the lines follow the real marks, then run the next cell."
  ];
  function send() {
    try { google.colab.kernel.invokeFunction("t5.set_points", [pts], {}); }
    catch (e) { msg.textContent = "Could not reach the notebook: " + e; }
  }
  function extended(p, q, colour, width) {
    var dx = q[0] - p[0], dy = q[1] - p[1], size = Math.sqrt(dx * dx + dy * dy) || 1;
    var ux = dx / size, uy = dy / size, far = W + H;
    ctx.strokeStyle = colour; ctx.lineWidth = width; ctx.beginPath();
    ctx.moveTo(p[0] - ux * far, p[1] - uy * far);
    ctx.lineTo(p[0] + ux * far, p[1] + uy * far);
    ctx.stroke();
  }
  function draw() {
    var scale = W / canvas.getBoundingClientRect().width;
    ctx.drawImage(img, 0, 0, W, H);
    if (pts.length >= 2) extended(pts[0], pts[1], COLOURS[0], 1.5 * scale);
    if (pts.length >= 4) extended(pts[2], pts[3], COLOURS[2], 1.5 * scale);
    for (var i = 0; i < pts.length; i++) {
      ctx.fillStyle = COLOURS[i]; ctx.beginPath();
      ctx.arc(pts[i][0], pts[i][1], 4 * scale, 0, 6.2832); ctx.fill();
      ctx.font = (14 * scale) + "px sans-serif";
      ctx.fillText((i < 2 ? "A" : "B") + ((i % 2) + 1),
                   pts[i][0] + 7 * scale, pts[i][1] - 7 * scale);
    }
    if (mouse && pts.length < 4) {
      ctx.strokeStyle = "rgba(255,255,255,0.9)"; ctx.lineWidth = 1 * scale;
      ctx.beginPath();
      ctx.moveTo(mouse[0], 0); ctx.lineTo(mouse[0], H);
      ctx.moveTo(0, mouse[1]); ctx.lineTo(W, mouse[1]);
      ctx.stroke();
    }
  }
  function refresh() { msg.textContent = PROMPTS[pts.length]; draw(); }
  function where(e) {
    var r = canvas.getBoundingClientRect();
    return [Math.round((e.clientX - r.left) * W / r.width),
            Math.round((e.clientY - r.top) * H / r.height)];
  }
  canvas.addEventListener("click", function (e) {
    if (pts.length >= 4) return;
    pts.push(where(e)); send(); refresh();
  });
  canvas.addEventListener("mousemove", function (e) {
    mouse = where(e); pos.textContent = "x = " + mouse[0] + ", y = " + mouse[1]; draw();
  });
  canvas.addEventListener("mouseleave", function () { mouse = null; draw(); });
  document.getElementById(uid + "u").onclick = function () { pts.pop(); send(); refresh(); };
  document.getElementById(uid + "r").onclick = function () { pts = []; send(); refresh(); };
  img.onload = refresh;
  img.src = "data:image/jpeg;base64,__IMAGE__";
})();
</script>
"""


def click_page(picture, show_width=1100):
    """The clickable picture as a page, with the frame embedded in it."""
    buffer = io.BytesIO()
    picture.save(buffer, format="JPEG", quality=90)
    page = CLICK_PAGE
    for token, value in (("__UID__", "t5" + uuid.uuid4().hex[:8]),
                         ("__W__", str(picture.width)),
                         ("__H__", str(picture.height)),
                         ("__SHOW__", str(min(show_width, picture.width))),
                         ("__IMAGE__", base64.b64encode(buffer.getvalue()).decode("ascii"))):
        page = page.replace(token, value)
    return page


if RUNNING_ON_COLAB:
    from google.colab import output
    from IPython.display import HTML, display

    def set_points(points):
        """Called from the picture each time a point is added or removed."""
        CLICKED_POINTS[:] = [(int(round(x)), int(round(y))) for x, y in points]

    output.register_callback("t5.set_points", set_points)
    display(HTML(click_page(FIRST_FRAME)))
else:
    print("Clicking works on Colab only. Here, use the grid above and type MARKS in Section 3.")

---

## 3. Settings (the only cell you edit)

**What this cell does:** holds every value that belongs to **your site and your decisions**. Nothing else in the notebook needs editing.

| Setting | Meaning |
|---|---|
| `MARKS` | Mark A and mark B, two points each `[(x, y), (x, y)]`. Leave it as `None` to use the four points you **clicked in Section 2**, or type them in yourself. A mark is a line **across** the walking direction. |
| `MARKS_METRES` | **D**: the distance between mark A and mark B, measured with a tape. Speed uses this. |
| `WALKWAY_LENGTH_M` | **W**: the length people must walk to clear the crossing (kerb to kerb). Clearance time uses this. Leave `None` to use D. |
| `MODEL_PATH` | The weights to use. Point it at your fine-tuned `best.pt` when you have one. |
| `CONFIDENCE` | Detections below this are never seen by the tracker. |
| `TRACK_BUFFER` | How many frames a lost person is remembered. 30 frames is one second at 30 fps. |
| `MATCH_THRESH` | How loosely boxes may match between frames. |
| `FLICKER_BELOW` | An ID seen in fewer frames than this is ignored (a doubtful extra box, not a person). |
| `MIN_WALKING_SPEED` | Slower than this, in m/s, counts as standing and is left out of the clearance time. |
| `MAX_SECONDS` | Longest video the service accepts. |

Write down the reason for each value you choose, because the report asks for it. The cell fails with a clear message in Section 10 if a mark or D is still `None`.

In [ ]:
# --- your site (from the tape and your clicks) ------------------------------
# Example: MARKS = {"A": [(400, 0), (400, 719)], "B": [(880, 0), (880, 719)]}
MARKS = {"A": None, "B": None}
MARKS_METRES = None        # D in metres, between mark A and mark B, from the tape
WALKWAY_LENGTH_M = None    # W in metres, kerb to kerb. None means use D

# --- the model ---------------------------------------------------------------
MODEL_PATH = WEIGHTS       # your fine-tuned best.pt goes here when you have one

# --- decisions you must be able to justify -----------------------------------
CONFIDENCE = 0.25
TRACK_BUFFER = 30
MATCH_THRESH = 0.8
FLICKER_BELOW = 5
MIN_WALKING_SPEED = 0.3    # m/s
MAX_SECONDS = 180

# --- use the points clicked in Section 2, unless MARKS was typed in above -----
if MARKS["A"] is None and MARKS["B"] is None and len(CLICKED_POINTS) == 4:
    MARKS = {"A": [CLICKED_POINTS[0], CLICKED_POINTS[1]],
             "B": [CLICKED_POINTS[2], CLICKED_POINTS[3]]}
    print("marks taken from your clicks in Section 2")

### Check your marks before tracking

Tracking takes a while, so confirm the marks first. Reading a pixel position off a grid is easy to get slightly wrong, and a wrong mark gives wrong speeds with no error message.

**What this cell does:** draws mark A and mark B on the **first, middle and last frames** of your clip, side by side, and prints their coordinates. Check three things:

1. **Each mark sits on the exact line you measured the tape from**, such as a stripe edge or your chalk line. Look at the ground under the red line, not at the people.
2. **The marks stay on the same piece of ground in all three pictures.** If the line drifts off its stripe edge, the camera moved.
3. **Mark A and mark B are the right way round and the right distance apart.** The gap between them should look like `MARKS_METRES` compared with the whole crossing.

If a mark is off, change its points in the cell above and run this cell again. It is quick.

In [ ]:
def extend_line(mark, length=5000):
    """Two far-apart points on the same line as the mark, so the whole line can be drawn."""
    (x1, y1), (x2, y2) = mark
    dx, dy = x2 - x1, y2 - y1
    size = (dx * dx + dy * dy) ** 0.5 or 1
    ux, uy = dx / size, dy / size
    return (x1 - ux * length, y1 - uy * length), (x1 + ux * length, y1 + uy * length)


def draw_marks(picture, marks, colour=(200, 40, 40)):
    """Each mark as a labelled line on a copy of the picture.

    The thin line is the whole line the notebook tests crossings against. The thick
    part is the stretch between the two points you gave.
    """
    canvas = picture.copy()
    draw = ImageDraw.Draw(canvas)
    for name, (start, end) in marks.items():
        draw.line(list(extend_line((start, end))), fill=colour, width=1)
        draw.line([tuple(start), tuple(end)], fill=colour, width=3)
        for x, y in (start, end):
            draw.ellipse([x - 5, y - 5, x + 5, y + 5], fill=colour)
        top = min((start, end), key=lambda point: point[1])
        draw.text((top[0] + 8, top[1] + 4), "mark %s" % name, fill=colour)
    return canvas


def side_by_side(pictures, width=1200):
    """Several pictures in one row, scaled to fit the width."""
    each = width // len(pictures)
    scaled = [picture.resize((each, int(picture.height * each / picture.width)))
              for picture in pictures]
    row = Image.new("RGB", (each * len(scaled), max(p.height for p in scaled)), "white")
    for index, picture in enumerate(scaled):
        row.paste(picture, (index * each, 0))
    return row


if MARKS["A"] is None or MARKS["B"] is None:
    print("Set MARKS in the settings cell above first, then run this cell again.")
    marks_preview = None
else:
    for name, (start, end) in MARKS.items():
        print("mark %s : from (%d, %d) to (%d, %d)" % (name, start[0], start[1], end[0], end[1]))
    check_frames = [0, CLIP["frames"] // 2, CLIP["frames"] - 1]
    print("frames shown:", ", ".join("%d (%.1f s)" % (i, i / CLIP["fps"]) for i in check_frames))
    marks_preview = side_by_side([
        draw_marks(as_picture(read_frame(VIDEO_IN, index)), MARKS) for index in check_frames])
marks_preview

---

## 4. Load the model once

**What this cell does:** loads the weights once, at startup, and reads the class names from the model itself (for us `person_left` and `person_right`), so nothing is typed by hand.

- If `best.pt` is missing it falls back to the stock model and says so. The stock model finds people but gives every one the same class, `person`, with no direction. Speeds still work.
- `PERSON_CLASS_IDS` lists the classes that start with `person`. Only those are tracked.
- This copy is loaded to read the classes and to prove the weights open. Each tracking run in Section 7 makes its **own fresh copy**, because the tracker keeps its memory inside the model object, and a reused one would carry the last run's people into the next run.

In [ ]:
if os.path.exists(MODEL_PATH):
    USING_TRAINED_MODEL = True
else:
    print("WARNING: no weights at %s" % MODEL_PATH)
    print("Falling back to the stock model. It finds people but cannot report direction.")
    MODEL_PATH = "yolo26n.pt"
    USING_TRAINED_MODEL = False

MODEL = YOLO(MODEL_PATH)
CLASS_NAMES = [MODEL.names[key] for key in sorted(MODEL.names)]
PERSON_CLASS_IDS = [index for index, name in enumerate(CLASS_NAMES)
                    if name.startswith("person")]

print("model   :", MODEL_PATH)
print("classes :", CLASS_NAMES)
print("tracked :", [CLASS_NAMES[index] for index in PERSON_CLASS_IDS])

---

## 5. Check the input before using it

A service that gets a bad file and answers with a stack trace has told the caller nothing. These three functions refuse bad input **before** any slow work, and every refusal says what was wrong and what to do.

**What this cell does:**

- `check_setup` refuses to run if a mark or D has not been set in Section 3.
- `validate_video` refuses: nothing sent, a missing or empty file, a file that is not a readable video, and a video longer than `MAX_SECONDS`.
- `validate_marks` refuses if a mark lies outside the video frame. This catches marks read from a video of a different size, for example 1080p marks used on a 4K clip.

Each returns `None` when everything is fine, or a dictionary with a `status` code, an `error` and a `detail`.

In [ ]:
def check_setup():
    """Refuse to run when the marks or the tape distance have not been filled in."""
    for name in ("A", "B"):
        if MARKS[name] is None:
            return {"status": 400, "error": "Mark %s has not been set." % name,
                    "detail": "Click its two points in Section 2 and run the settings "
                              "cell again, or type MARKS in Section 3."}
        (x1, y1), (x2, y2) = MARKS[name]
        if ((x2 - x1) ** 2 + (y2 - y1) ** 2) ** 0.5 < 40:
            return {"status": 400,
                    "error": "The two points of mark %s are too close together." % name,
                    "detail": "Put them at the near and far ends of the line, so the "
                              "direction of the line is accurate."}
    if not MARKS_METRES or MARKS_METRES <= 0:
        return {"status": 400, "error": "The distance between the marks is not set.",
                "detail": "Set MARKS_METRES in Section 3 to the tape measurement, in metres."}
    return None


def validate_video(path):
    """Check that what arrived is a readable video of a sensible length.

    Returns (info, error), exactly one of which is None.
    """
    if path is None:
        return None, {"status": 400, "error": "No video was sent.",
                      "detail": "Attach an MP4 of the crossing and send it again."}
    path = str(path)
    if not os.path.exists(path):
        return None, {"status": 400, "error": "The video file was not found.",
                      "detail": "No file at '%s'." % os.path.basename(path)}
    if os.path.getsize(path) == 0:
        return None, {"status": 400, "error": "The upload was empty.",
                      "detail": "The file '%s' is zero bytes." % os.path.basename(path)}
    info = video_info(path)
    if info is None or info["frames"] <= 0 or info["fps"] <= 0:
        return None, {"status": 415, "error": "That file is not a readable video.",
                      "detail": "Send an MP4 recorded on a phone."}
    seconds = info["frames"] / info["fps"]
    if seconds > MAX_SECONDS:
        return None, {"status": 413, "error": "The video is too long to process.",
                      "detail": "%.0f seconds is beyond the limit of %d seconds."
                                % (seconds, MAX_SECONDS)}
    return info, None


def validate_marks(info):
    """Refuse marks that fall outside this video's frame."""
    for name, points in MARKS.items():
        for x, y in points:
            if not (0 <= x <= info["width"] and 0 <= y <= info["height"]):
                return {"status": 400,
                        "error": "Mark %s lies outside the video frame." % name,
                        "detail": "The point (%d, %d) is outside %d x %d. The marks were "
                                  "probably read from a different video or camera position."
                                  % (x, y, info["width"], info["height"])}
    return None

---

## 6. Geometry: when did the feet cross a mark

**What this cell does:**

- `signed_side` is the cross product test from the marking-regions lab. It is positive on one side of a mark's line and negative on the other. Which side is positive does not matter. What matters is that it **changes sign** when someone crosses.
- `crossing_time` walks through one person's observations, finds the pair of frames where the sign changes, and **interpolates** between them. If the feet were 3 pixels before the line in one frame and 9 pixels past it in the next, the crossing is placed a quarter of the way between the frames. So the time is not rounded to a whole frame, which matters because speed divides by a short time.

It returns the crossing time in seconds, or `None` if the feet never crossed that mark.

In [ ]:
def signed_side(point, mark):
    """Cross product: positive on one side of the mark, negative on the other.

    Its size is proportional to the distance from the mark's line, which lets us
    interpolate the exact moment of crossing between two frames.
    """
    (x1, y1), (x2, y2) = mark
    return (x2 - x1) * (point[1] - y1) - (y2 - y1) * (point[0] - x1)


def crossing_time(observations, mark):
    """Seconds at which the feet first cross the mark, or None if they never do."""
    for before, after in zip(observations, observations[1:]):
        d1 = signed_side(before["foot"], mark)
        d2 = signed_side(after["foot"], mark)
        if d1 == 0:
            return before["seconds"]
        if d1 * d2 < 0:
            fraction = abs(d1) / (abs(d1) + abs(d2))
            return before["seconds"] + fraction * (after["seconds"] - before["seconds"])
    return None

---

## 7. Tracking

A detector has no memory: it gives boxes in each frame and nothing says which box is the same person as in the last frame. **ByteTrack** adds that. It predicts where each known person should be, matches new boxes to those predictions, and remembers a person who disappears for a short while. `persist=True` keeps that memory from one frame to the next.

**What this cell does (tracker settings):** ByteTrack's settings live in a small YAML file. `tracker_settings` copies the default file with your changes (`track_buffer`, `match_thresh`) and returns its path. `TRACKER_FILE` is the file used for every run unless Section 15 tries others.

In [ ]:
DEFAULT_SETTINGS_FILE = os.path.join(str(ULTRALYTICS_ROOT), "cfg", "trackers",
                                     "bytetrack.yaml")
with open(DEFAULT_SETTINGS_FILE, encoding="utf-8") as handle:
    DEFAULT_SETTINGS = yaml.safe_load(handle)


def tracker_settings(name, **changes):
    """Write a copy of the ByteTrack settings with some values changed."""
    unknown = set(changes) - set(DEFAULT_SETTINGS)
    if unknown:
        raise KeyError("ByteTrack has no setting called %s" % ", ".join(sorted(unknown)))
    settings = dict(DEFAULT_SETTINGS)
    settings.update(changes)
    path = os.path.join(WORK_DIR, name + ".yaml")
    with open(path, "w", encoding="utf-8") as handle:
        yaml.safe_dump(settings, handle)
    return path


TRACKER_FILE = tracker_settings("t5_tracker", track_buffer=TRACK_BUFFER,
                                match_thresh=MATCH_THRESH)
print("tracker settings file:", TRACKER_FILE)
print("track_buffer = %d frames, match_thresh = %.2f" % (TRACK_BUFFER, MATCH_THRESH))

**What this cell does (`track_video`):** reads the video **one frame at a time**, runs the detector and tracker on each frame, and collects what is reported for every person ID. Reading frame by frame keeps memory flat on a long clip and gives every observation its **frame number and time in seconds**, which speed needs.

For each observation it stores:

- the frame number and the time in seconds (frame number divided by frame rate)
- the box
- the **foot point**: the middle of the bottom edge of the box. Feet are on the ground, where the marks are. The box centre would move when someone swings an arm or carries a bag.
- the class the model gave and its confidence

A box with no ID is one the tracker has not yet accepted as a person (usually the first frame someone appears), so it is left out. The function returns a dictionary from track ID to a list of observations in time order.

In [ ]:
def track_video(path, tracker_file, confidence=CONFIDENCE, show_progress=True):
    """Run the tracker over a video and collect what it reports for every identity."""
    # A fresh model for every run: the tracker lives inside the model object.
    model = YOLO(MODEL_PATH)

    video = cv2.VideoCapture(path)
    fps = video.get(cv2.CAP_PROP_FPS)
    total = int(video.get(cv2.CAP_PROP_FRAME_COUNT))
    tracks = {}
    index = -1

    while True:
        ok, frame = video.read()
        if not ok:
            break
        index = index + 1

        result = model.track(frame, persist=True, tracker=tracker_file,
                             conf=confidence, classes=PERSON_CLASS_IDS,
                             verbose=False)[0]

        if result.boxes is not None and result.boxes.id is not None:
            for box, track_id, class_id, score in zip(
                    result.boxes.xyxy.tolist(), result.boxes.id.int().tolist(),
                    result.boxes.cls.int().tolist(), result.boxes.conf.tolist()):
                left, top, right, bottom = box
                tracks.setdefault(track_id, []).append({
                    "frame": index,
                    "seconds": round(index / fps, 3),
                    "box": [round(value, 1) for value in box],
                    "foot": [round((left + right) / 2, 1), round(bottom, 1)],
                    "class_name": CLASS_NAMES[class_id],
                    "confidence": round(score, 3),
                })

        if show_progress and index % 200 == 0:
            print("  frame %d of %d" % (index, total))

    video.release()
    return tracks

---

## 8. Speed and clearance time

**What this cell does:**

- `majority_class` gives a person the class the model gave them most often (the class can flicker between frames, so one vote per frame and the most common wins).
- `measure_speeds` goes through every person who lasted at least `FLICKER_BELOW` frames and finds their crossing time at mark A and mark B. For everyone who crossed **both**, it computes **speed = D / |time at B − time at A|** and records which way they went (`A to B` or `B to A`), whether they count as walking (`speed >= MIN_WALKING_SPEED`), and `known_at`, the moment their second crossing happened. The output video uses `known_at` so that a speed appears on screen only from the moment it could really be known.
- Someone who crossed only one mark, or neither, goes on an **excluded** list with the reason. They are never given an invented speed.
- `summarise` turns the rows into the numbers for the report: how many people were measured, the average and **slowest** walking speed, and the **clearance time = W / slowest walking speed**.

In [ ]:
def majority_class(observations):
    """The class the model gave this person most often."""
    return Counter(o["class_name"] for o in observations).most_common(1)[0][0]


def measure_speeds(tracks, marks, metres_between_marks):
    """One row per person who crossed both marks, plus a list of those left out."""
    rows, excluded = [], []
    for track_id in sorted(tracks):
        observations = tracks[track_id]
        if len(observations) < FLICKER_BELOW:
            continue  # a flicker, not a person
        t_a = crossing_time(observations, marks["A"])
        t_b = crossing_time(observations, marks["B"])
        if t_a is None or t_b is None:
            if t_a is None and t_b is None:
                which = "neither mark"
            else:
                which = "only mark A" if t_b is None else "only mark B"
            excluded.append({"id": track_id,
                             "reason": "crossed %s (fragment, or left the frame?)" % which})
            continue
        seconds = abs(t_b - t_a)
        if seconds == 0:
            excluded.append({"id": track_id, "reason": "crossed both marks at the same instant"})
            continue
        speed = metres_between_marks / seconds
        rows.append({
            "id": track_id,
            "class": majority_class(observations),
            "order": "A to B" if t_b > t_a else "B to A",
            "seconds": round(seconds, 2),
            "speed_mps": round(speed, 2),
            "walking": speed >= MIN_WALKING_SPEED,
            "known_at": max(t_a, t_b),
        })
    return rows, excluded


def summarise(rows, length_m):
    """The numbers for the report: speeds and the clearance time at the slowest walker."""
    walkers = [r for r in rows if r["walking"]]
    summary = {
        "people_measured": len(rows),
        "walking": len(walkers),
        "standing": len(rows) - len(walkers),
        "walkway_length_m": length_m,
        "average_walking_mps": None,
        "slowest_walking_mps": None,
        "clearance_seconds": None,
    }
    if walkers:
        speeds = [r["speed_mps"] for r in walkers]
        slowest = min(speeds)
        summary["average_walking_mps"] = round(sum(speeds) / len(speeds), 2)
        summary["slowest_walking_mps"] = slowest
        summary["clearance_seconds"] = round(length_m / slowest, 1)
    return summary

---

## 9. Draw the output video

**What this cell does (colours and labels):** gives every lasting ID its own colour (IDs jump, like 2, 12, 40, so colouring by number would repeat colours), and defines `draw_label`, which draws text on a filled coloured strip so it stays readable on any background. The strip goes above the box, or just inside it if the box is at the top of the frame.

In [ ]:
TRACK_COLOURS = [
    (214, 69, 65), (46, 112, 196), (72, 160, 88), (222, 158, 54),
    (132, 94, 194), (64, 172, 190), (206, 118, 168), (128, 104, 82),
    (40, 40, 40), (150, 150, 150),
]
FONT = cv2.FONT_HERSHEY_SIMPLEX


def track_colours(tracks):
    """One colour (RGB) per lasting identity, in the order they appeared."""
    lasting = [t for t in sorted(tracks) if len(tracks[t]) >= FLICKER_BELOW]
    return {track_id: TRACK_COLOURS[index % len(TRACK_COLOURS)]
            for index, track_id in enumerate(lasting)}


def draw_label(frame, text, left, top, bgr, scale, thickness):
    """Text on a filled strip, above the point (left, top) when there is room."""
    (width, height), baseline = cv2.getTextSize(text, FONT, scale, thickness)
    strip_top = top - height - baseline - 4
    if strip_top < 0:
        strip_top = top + 2
    cv2.rectangle(frame, (left, strip_top),
                  (left + width + 6, strip_top + height + baseline + 4), bgr, -1)
    cv2.putText(frame, text, (left + 3, strip_top + height + 1), FONT, scale,
                (255, 255, 255), thickness, cv2.LINE_AA)

**What this cell does (`write_video`):** reads the input video again and writes a new one. On every frame it draws:

- **mark A and mark B** as red lines
- for every tracked person, a **box in their colour** with a label `ID 3 | person_right | 1.42 m/s`. Before the person has crossed both marks the label shows `...` instead of a speed. A person slower than `MIN_WALKING_SPEED` is labelled `(standing)`.
- a **panel in the top left** with the people measured so far, the slowest walking speed so far and the clearance time so far. It uses only speeds already known at that moment, so it grows as people cross, instead of showing the final answer from the first frame.

Text size scales with the video height, so it is readable on 720p and 1080p. OpenCV writes a video that many browsers cannot play, so the last step converts it to H.264 with ffmpeg (Colab has it). Without ffmpeg the raw file is kept.

In [ ]:
def write_video(path, tracks, rows, marks, length_m, info, out_path):
    """Write a copy of the video with marks, boxes, IDs and speeds drawn on."""
    colours = track_colours(tracks)
    speed_of = {row["id"]: row for row in rows}
    observations_at = {}
    for track_id in colours:
        for observation in tracks[track_id]:
            observations_at.setdefault(observation["frame"], []).append(
                (track_id, observation))

    fps, width, height = info["fps"], info["width"], info["height"]
    unit = height / 540.0
    scale = 0.5 * unit
    thickness = max(1, round(unit))

    raw = out_path.replace(".mp4", "_raw.mp4")
    writer = cv2.VideoWriter(raw, cv2.VideoWriter_fourcc(*"mp4v"), fps, (width, height))
    video = cv2.VideoCapture(path)
    index = -1

    while True:
        ok, frame = video.read()
        if not ok:
            break
        index = index + 1
        seconds = index / fps

        for name, (start, end) in marks.items():
            far_a, far_b = extend_line((start, end))
            cv2.line(frame, (int(far_a[0]), int(far_a[1])), (int(far_b[0]), int(far_b[1])),
                     (40, 40, 220), thickness * 2)
            top = min((start, end), key=lambda point: point[1])
            draw_label(frame, "mark " + name, int(top[0]) + 6, int(top[1]) + 28 * thickness,
                       (40, 40, 220), scale, thickness)

        for track_id, observation in observations_at.get(index, []):
            bgr = colours[track_id][::-1]
            left, top, right, bottom = [int(value) for value in observation["box"]]
            cv2.rectangle(frame, (left, top), (right, bottom), bgr, thickness * 2)
            row = speed_of.get(track_id)
            if row is not None and row["known_at"] <= seconds:
                speed_text = "%.2f m/s" % row["speed_mps"]
                if not row["walking"]:
                    speed_text += " (standing)"
            else:
                speed_text = "..."
            draw_label(frame, "ID %d | %s | %s" % (track_id, majority_class(tracks[track_id]),
                                                   speed_text),
                       left, top, bgr, scale, thickness)

        known = summarise([r for r in rows if r["known_at"] <= seconds], length_m)
        slowest = known["slowest_walking_mps"]
        lines = [
            "people measured: %d" % known["people_measured"],
            "slowest walking: %s" % ("-" if slowest is None else "%.2f m/s" % slowest),
            "clearance (%.1f m): %s" % (length_m, "-" if slowest is None
                                        else "%.1f s" % known["clearance_seconds"]),
        ]
        line_height = int(26 * unit)
        panel_width = int(max(cv2.getTextSize(t, FONT, scale, thickness)[0][0]
                              for t in lines) + 16)
        cv2.rectangle(frame, (8, 8), (8 + panel_width, 8 + line_height * len(lines) + 8),
                      (30, 30, 30), -1)
        for number, text in enumerate(lines):
            cv2.putText(frame, text, (16, 8 + line_height * (number + 1)), FONT, scale,
                        (255, 255, 255), thickness, cv2.LINE_AA)

        writer.write(frame)

    video.release()
    writer.release()

    if shutil.which("ffmpeg"):
        subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", raw,
                        "-vcodec", "libx264", "-pix_fmt", "yuv420p", out_path], check=True)
        os.remove(raw)
    else:
        os.replace(raw, out_path)
    return out_path

---

## 10. The whole service in one function

**What this cell does:** `process_video` is the whole service in one function, the same idea as `build_reply` in Lesson 4. Given a video it:

1. checks the setup, the video and the marks (Section 5), and stops with a clear error if any fail
2. tracks every person (Section 7)
3. measures speeds and the clearance time (Section 8)
4. writes the output video (Section 9)
5. returns a **reply dictionary**, the output video path and the raw tracks

The reply is built in one place and is always a dictionary. A reply with `"error"` failed, and one with `"summary"` did not. An empty crossing with nobody in it is a valid answer of zero people, not an error.

The Gradio page and the plain run in Section 11 both call this function and nothing else, so the page can be replaced without touching the logic.

In [ ]:
def process_video(path, output_path=VIDEO_OUT):
    """Video in, annotated video and speeds out. Returns (reply, output path, tracks)."""
    error = check_setup()
    if error is not None:
        return error, None, None
    info, error = validate_video(path)
    if error is not None:
        return error, None, None
    error = validate_marks(info)
    if error is not None:
        return error, None, None

    started = time.time()
    length_m = WALKWAY_LENGTH_M or MARKS_METRES
    marks = {name: [tuple(point) for point in points] for name, points in MARKS.items()}

    tracks = track_video(str(path), TRACKER_FILE)
    rows, excluded = measure_speeds(tracks, marks, MARKS_METRES)
    summary = summarise(rows, length_m)
    write_video(str(path), tracks, rows, marks, length_m, info, output_path)
    elapsed = time.time() - started

    reply = {
        "status": 200,
        "summary": summary,
        "people": [{key: value for key, value in row.items() if key != "known_at"}
                   for row in rows],
        "excluded": excluded,
        "identities_produced": len(tracks),
        "marks_metres": MARKS_METRES,
        "model": MODEL_PATH,
        "trained_model": USING_TRAINED_MODEL,
        "video_seconds": round(info["frames"] / info["fps"], 1),
        "processing_seconds": round(elapsed, 1),
        "output_video": output_path,
    }
    print("processed %.1f s of video in %.1f s: %d people measured"
          % (reply["video_seconds"], elapsed, summary["people_measured"]))
    return reply, output_path, tracks

---

## 11. Run it on your clip

**What this cell does:** runs `process_video` on your clip (`VIDEO_IN`) and prints the result as a table:

- one row per measured person: ID, class, direction of travel, seconds between the marks, speed
- anyone **excluded**, with the reason. Check this list: slow walkers who do not finish before the clip ends will be here, and leaving them out makes the "slowest speed" look faster than it really is
- the summary: number of people, average and slowest walking speed, and the clearance time

It also saves `speeds.csv` (the table) and `results.json` (everything) to `artifacts/T5/` for your report.

In [ ]:
def print_report(reply):
    """The reply as a table, or the error that came back."""
    if "error" in reply:
        print("ERROR %d: %s %s" % (reply["status"], reply["error"], reply["detail"]))
        return
    print("%5s %-13s %-8s %8s %8s" % ("id", "class", "order", "seconds", "m/s"))
    for row in reply["people"]:
        print("%5d %-13s %-8s %8.2f %8.2f%s"
              % (row["id"], row["class"], row["order"], row["seconds"], row["speed_mps"],
                 "" if row["walking"] else "   standing"))
    for item in reply["excluded"]:
        print("  excluded id %d: %s" % (item["id"], item["reason"]))
    summary = reply["summary"]
    print("")
    print("people measured      : %d (%d walking, %d below %.1f m/s treated as standing)"
          % (summary["people_measured"], summary["walking"], summary["standing"],
             MIN_WALKING_SPEED))
    if summary["slowest_walking_mps"] is None:
        print("nobody crossed both marks as a walker, so no speed or clearance time.")
    else:
        print("average walking speed: %.2f m/s" % summary["average_walking_mps"])
        print("slowest walking speed: %.2f m/s" % summary["slowest_walking_mps"])
        print("clearance time for %.1f m at the slowest speed: %.1f s"
              % (summary["walkway_length_m"], summary["clearance_seconds"]))


def save_results(reply):
    """Write speeds.csv and results.json into the work folder."""
    with open(os.path.join(WORK_DIR, "results.json"), "w", encoding="utf-8") as handle:
        json.dump(reply, handle, indent=1)
    with open(os.path.join(WORK_DIR, "speeds.csv"), "w", newline="", encoding="utf-8") as handle:
        writer = csv.writer(handle)
        writer.writerow(["id", "class", "order", "seconds", "speed_mps", "walking"])
        for row in reply["people"]:
            writer.writerow([row["id"], row["class"], row["order"], row["seconds"],
                             row["speed_mps"], row["walking"]])


REPLY, OUTPUT_VIDEO, TRACKS = process_video(VIDEO_IN)
print("")
print_report(REPLY)
if "error" not in REPLY:
    save_results(REPLY)
    print("")
    print("saved speeds.csv and results.json in", WORK_DIR)
    print("output video:", OUTPUT_VIDEO)

**What this cell does:** plays the output video inside the notebook. `show_video` shrinks a copy to 640 pixels wide and embeds it in the page. The full-size demo video is the file at `VIDEO_OUT`, which is the one to hand in. Watch it: every person should keep **one ID** from entering to leaving, and the speed should appear after they pass the second mark.

In [ ]:
from IPython.display import HTML


def show_video(path, width=640):
    """Play a video inside the notebook, from a smaller copy of the file."""
    if not shutil.which("ffmpeg"):
        print("No ffmpeg here. Open %s instead." % path)
        return None
    small = path.replace(".mp4", "_small.mp4")
    subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", path,
                    "-vf", "scale=%d:-2" % width, "-vcodec", "libx264",
                    "-pix_fmt", "yuv420p", small], check=True)
    with open(small, "rb") as handle:
        data = base64.b64encode(handle.read()).decode("ascii")
    return HTML('<video width="%d" controls><source src="data:video/mp4;base64,%s" '
                'type="video/mp4"></video>' % (width, data))


show_video(OUTPUT_VIDEO) if OUTPUT_VIDEO else None

---

## 12. Check the tracks by looking at them

Do not trust the speed table until you have looked at this. A speed is only correct if each person had **one ID** between the two marks. If the tracker lost someone and gave them a new ID in the middle, that person crossed each mark under a different ID, so they are excluded and their speed is lost. If two people swapped IDs, a speed can belong to the wrong person.

**What this cell does:**

- `describe_tracks` lists every ID: how many frames it lasted, when it started and ended, its class, and whether it is a flicker (too short to be a person).
- `draw_tracks` draws every lasting person's path along the ground on the first frame, one colour per ID, with a large dot at the start. It also draws mark A and mark B.

How to read the picture: **a colour that changes part way along one continuous path** is one person who became two IDs (a fragment). **Two paths that swap colour where they cross** are two people who swapped IDs. The marks should sit where the paths are clean and continuous. If a fragment happens between the marks, try a larger `TRACK_BUFFER` (Section 15) or move the marks.

In [ ]:
def describe_tracks(tracks):
    print("%5s %6s %8s %8s %-13s" % ("id", "frames", "from s", "to s", "class"))
    for track_id in sorted(tracks):
        observations = tracks[track_id]
        print("%5d %6d %8.1f %8.1f %-13s %s"
              % (track_id, len(observations), observations[0]["seconds"],
                 observations[-1]["seconds"], majority_class(observations),
                 "flicker" if len(observations) < FLICKER_BELOW else ""))


def draw_tracks(background, tracks, marks, title):
    """Every lasting track as a coloured path through the feet, with the marks."""
    canvas = Image.blend(background, Image.new("RGB", background.size, "white"), 0.6)
    draw = ImageDraw.Draw(canvas)
    colours = track_colours(tracks)
    for track_id, colour in colours.items():
        points = [tuple(observation["foot"]) for observation in tracks[track_id]]
        if len(points) > 1:
            draw.line(points, fill=colour, width=3)
        x, y = points[0]
        draw.ellipse([x - 6, y - 6, x + 6, y + 6], fill=colour)
        draw.text((x + 8, y - 16), "ID %d" % track_id, fill=colour)
    for name, (start, end) in marks.items():
        draw.line(list(extend_line((start, end))), fill=(200, 40, 40), width=3)
        draw.text((start[0] + 6, start[1] + 4), "mark %s" % name, fill=(200, 40, 40))
    draw.rectangle([0, 0, canvas.width, 34], fill=(255, 255, 255))
    draw.text((8, 4), title, fill=(30, 30, 30))
    draw.text((8, 18), "%d identities, %d of them flickers not drawn. The large dot is where "
              "each one started." % (len(tracks), len(tracks) - len(colours)),
              fill=(90, 90, 90))
    return canvas


if TRACKS:
    describe_tracks(TRACKS)
    marks_as_points = {name: [tuple(point) for point in points] for name, points in MARKS.items()}
    display_picture = draw_tracks(FIRST_FRAME, TRACKS, marks_as_points,
                                  "ByteTrack, track_buffer=%d, match_thresh=%.2f"
                                  % (TRACK_BUFFER, MATCH_THRESH))
else:
    print("Nothing was tracked. See the message printed in Section 11.")
    display_picture = None
display_picture

---

## 13. A web page with a public link

Lesson 4 turned the model into something a stranger could use. This does the same for the video demo.

**What this cell does:** wraps `process_video` in a Gradio page: an upload box for the video, and two outputs, the **annotated video** and the **reply** as structured data. `predict` is the function the page calls. It returns the annotated video, or nothing and the error when something was refused. `share=True` makes the public link; without it the page only works inside this runtime.

Things to know before putting the link in a report:

- The link lives for 72 hours and stops when this runtime stops.
- Anyone holding it can send videos to it.
- The marks and D belong to **your camera position**, not to the request. A video filmed from a different position needs new marks, so only upload videos from the same setup. Otherwise `validate_marks` may refuse it, or the speeds may be wrong.

In [ ]:
import gradio as gr


def predict(video_path):
    """What the page calls. Returns the annotated video and the reply."""
    reply, output_path, _ = process_video(video_path)
    return output_path, reply


demo = gr.Interface(
    fn=predict,
    inputs=[gr.Video(label="Crossing video")],
    outputs=[gr.Video(label="Boxes, IDs and speeds"), gr.JSON(label="Reply")],
    title="Pedestrian speed and clearance time",
    description="Upload a video of the crossing, filmed from the same camera position as "
                "the marks. The reply gives each person's speed and the clearance time.",
)

demo.launch(share=True)

---

## 14. Error tests

Like Lesson 4, what is being checked is not whether the service survives, but whether what comes back **tells the caller what to do next**. These call `process_video` directly, because a widget that refuses a bad file is not the same as a function that defends itself.

**What this cell does:** builds six inputs in a temporary folder and sends each through `process_video`:

1. nothing sent
2. a text file
3. an empty file
4. a video of a different size from yours, which the marks do not fit
5. a video with nobody in it, which must be a **valid answer of zero people** (status 200), not an error
6. a video that is too long (`MAX_SECONDS` is lowered for this one test and put back)

The table is printed at the end in one piece. Copy the status and message for each into your report **as they came back**.

In [ ]:
import tempfile

scratch = tempfile.mkdtemp()


def blank_video(name, width, height, frames, fps=30):
    """A video of an empty grey scene."""
    path = os.path.join(scratch, name)
    writer = cv2.VideoWriter(path, cv2.VideoWriter_fourcc(*"mp4v"), fps, (width, height))
    grey = np.full((height, width, 3), 200, np.uint8)
    for _ in range(frames):
        writer.write(grey)
    writer.release()
    return path


text_file = os.path.join(scratch, "not_a_video.txt")
with open(text_file, "w", encoding="utf-8") as handle:
    handle.write("this is not a video")

empty_file = os.path.join(scratch, "empty.mp4")
open(empty_file, "wb").close()

other_size = blank_video("other_size.mp4", 320, 240, 30)
nobody = blank_video("nobody.mp4", CLIP["width"], CLIP["height"], 60, fps=CLIP["fps"])
too_long = blank_video("too_long.mp4", CLIP["width"], CLIP["height"], 60, fps=CLIP["fps"])

CASES = [
    ("nothing sent", None),
    ("a text file", text_file),
    ("an empty file", empty_file),
    ("a video of a different size", other_size),
    ("a video with nobody in it", nobody),
    ("a video that is too long", too_long),
]

rows = []
saved_limit = MAX_SECONDS
for label, payload in CASES:
    MAX_SECONDS = 1 if label == "a video that is too long" else saved_limit
    reply, _, _ = process_video(payload, output_path=os.path.join(scratch, "out.mp4"))
    if "error" in reply:
        summary = reply["error"]
    else:
        summary = "%d people measured" % reply["summary"]["people_measured"]
    rows.append((label, reply["status"], summary))
MAX_SECONDS = saved_limit

print("")
print("%-30s %-7s %s" % ("case", "status", "what came back"))
for label, status, summary in rows:
    print("%-30s %-7s %s" % (label, status, summary))

---

## 15. Tune the tracker (optional, but the report asks for it)

The lab asks you to find where each tracker setting **breaks on your clip** and write the value down. Section 7's table explains the two settings:

- `track_buffer`: too low and someone who walks behind a pole comes out as a new person. Too high and someone who left can be handed to a newcomer.
- `match_thresh`: too strict and people fragment. Too loose and two people can share one ID.

**What this cell does:** re-runs the tracker on your clip with several values of each setting, one at a time, and prints for each:

- the number of lasting identities (compare with the true number of people you can count in the video)
- how many people were **measured** (crossed both marks), which is the number T5 cares about
- how many were excluded because they crossed only one mark, a sign of fragments
- the slowest walking speed, since clearance time depends on it

It takes about as long as eight runs of the tracker, so skip it on a slow runtime. Look for the **smallest** `track_buffer` that gives the right number of measured people, and for the first value, if any, where new trouble appears (a speed that looks wrong on the video). Leave `match_thresh` at the default unless the path picture in Section 12 shows a specific problem it fixes.

In [ ]:
SWEEPS = [("track_buffer", (1, 5, 30, 90)), ("match_thresh", (0.3, 0.6, 0.8, 0.95))]
marks_as_points = {name: [tuple(point) for point in points] for name, points in MARKS.items()}

for setting, values in SWEEPS:
    print("%-14s %10s %9s %9s %14s" % (setting, "identities", "measured", "excluded",
                                       "slowest m/s"))
    for value in values:
        tracker_file = tracker_settings("sweep_%s_%s" % (setting, value), **{setting: value})
        sweep_tracks = track_video(VIDEO_IN, tracker_file, show_progress=False)
        sweep_rows, sweep_excluded = measure_speeds(sweep_tracks, marks_as_points, MARKS_METRES)
        lasting = sum(1 for o in sweep_tracks.values() if len(o) >= FLICKER_BELOW)
        walkers = [r["speed_mps"] for r in sweep_rows if r["walking"]]
        print("%-14s %10d %9d %9d %14s"
              % (value, lasting, len(sweep_rows), len(sweep_excluded),
                 "%.2f" % min(walkers) if walkers else "-"))
    print("")

---

## 16. Before leaving

- The output video was **watched**: every person keeps one ID, and each speed appears after the second mark.
- The path picture in Section 12 was looked at, not only the numbers. Fragments between the marks are explained or fixed.
- The marks were clicked on the real lines in Section 2 and checked on the first, middle and last frames in Section 3, and sit on the lines the tape was measured from.
- The camera check in Section 2 came out low, or the picture was looked at and the reason is known.
- The **excluded** list was read, especially slow walkers who did not finish before the clip ended.
- `MARKS_METRES` (D) and `WALKWAY_LENGTH_M` (W) are the tape measurements from the scene, with a photo of the tape.
- You can say why you chose `MIN_WALKING_SPEED`, `CONFIDENCE`, and the marks' positions, and where `track_buffer` and `match_thresh` broke on your clip.
- The error table from Section 14 is copied into the report as it came back.
- Fine-tuned weights, if any: compare the number of people measured before and after. That is the comparison that matters for T5.
- Staged walks, if any, are labelled as staged in the report.

**Known limits to state in the report:**

- A person walking at an angle covers more ground than D, so their speed reads low. Side-on filming keeps this small.
- One person's speed is the average between the two marks, not their speed at any single moment.
- A clearance time built from the slowest walker in one clip is only as good as the slowest person who happened to be in that clip.

The cell below closes the public link from Section 13. Run it when you finish, or the runtime keeps serving.

In [ ]:
demo.close()
print("The share link is closed.")